In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read DQ-passed records only
df = spark.table("job_market.silver.jobs_validated")

print("Validated records:", df.count())

display(df.limit(10))

Validated records: 2221


source company ingestion_timestamp ingestion_date job_id job_title department office team job_level employment_type location location_normalized all_locations workplace_type is_remote description posted_at updated_at salary_min salary_max salary_currency salary_interval job_url apply_url quality_status quality_reason dq_processed_at greenhouse Cloudflare 2026-10-06T12:44:37.740Z 2026-10-06 5390243 Senior Software Engineer, Durable Objects (DO) Emerging Technology and Incubation Amsterdam, Netherlands null null null In-Office in-office List(In-Office) null false <div class="content-intro"><h3>About Us</h3>
<p>At Cloudflare, we are on a mission to help build a better Internet. Today the company runs one of the world’s largest networks that powers millions of websites and other Internet properties for customers ranging from individual bloggers to SMBs to Fortune 500 companies. Cloudflare protects and accelerates any Internet application online without adding hardware, installing software, or changing a line of code. Internet properties powered by Cloudflare all have web traffic routed through its intelligent global network, which gets smarter with every request. As a result, they see significant improvement in performance and a decrease in spam and other attacks. Cloudflare was named to Entrepreneur Magazine’s Top Company Cultures list and ranked among the World’s Most Innovative Companies by Fast Company.</p>
<p>At Cloudflare, we’re not looking for people who wait for a polished roadmap; we’re looking for the builders who see the cracks in the Internet that everyone else has simply learned to live with. We value candidates who have the instinct to spot a "normalized" problem and the AI-native curiosity to create a solution using the latest tools. Our culture is built on iteration, leveraging AI to ship faster today to make it better tomorrow, while ensuring that every improvement, no matter how small, is shared across the team to lift everyone up. If you’re the type of person who values curiosity over bureaucracy, and that AI is a partner in solving tough problems to keep the Internet moving forward, you’ll fit right in.</p></div><h3><strong>Available Locations: Austin, TX</strong></h3>
<h3>About the Role</h3>
<p>Cloudflare’s Emerging Technologies &amp; Incubation team builds and launches bold new products on Cloudflare’s edge network. This role is on the Durable Objects team, which powers stateful serverless applications for Cloudflare customers. You’ll help evolve the runtime and the low-level routing and storage systems that support real-time chat, multiplayer games, AI agents, and other stateful workloads at scale.</p>
<h3>Responsibilities</h3>
<ul>
<li>Build and improve the Durable Objects runtime.</li>
<li>Design and develop the low-level routing and storage systems that power customer applications.</li>
<li>Improve reliability, scalability, performance, and feature depth.</li>
<li>Own code end-to-end from design through production.</li>
<li>Solve complex problems in production distributed systems.</li>
<li>Work across JavaScript, TypeScript, Rust, and C++.</li>
<li>Help drive the continued evolution of the system.</li>
</ul>
<h3>Desirable Skills, Knowledge &amp; Experience</h3>
<ul>
<li>5+ years of experience writing systems software.</li>
<li>Operational mindset with experience owning code in production.</li>
<li>Strong foundation in data structures, algorithms, and system design.</li>
<li>Deep expertise in at least one systems programming language such as C/C++ or Rust.</li>
<li>Experience designing and architecting large-scale distributed systems.</li>
<li>Familiarity with cloud platforms, especially serverless architectures.</li>
<li>Experience as a technical leader or team lead, driving engineering decisions.</li>
</ul>
<h3>Bonus Points</h3>
<ul>
<li>Experience working on database internals, especially SQLite.</li>
<li>Deep understanding of web technologies such as browsers, HTTP, JavaScript, and WebAssembly.</li>
<l

In [0]:
# ============================================================
# DUPLICATE JOB ID CHECK
# ============================================================

duplicate_job_ids = (
    df
    .groupBy("source", "job_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print("Duplicate source + job_id combinations:")

display(duplicate_job_ids)

Duplicate source + job_id combinations:


source,job_id,count


In [0]:
duplicate_id_count = duplicate_job_ids.count()

print("Number of duplicate source + job_id groups:", duplicate_id_count)

Number of duplicate source + job_id groups: 0


In [0]:
# ============================================================
# RECORD RANKING
# ============================================================

window_spec = (
    Window
    .partitionBy("source", "job_id")
    .orderBy(
        F.col("posted_at").desc_nulls_last(),
        F.col("updated_at").desc_nulls_last(),
        F.col("ingestion_timestamp").desc_nulls_last()
    )
)

ranked_df = (
    df
    .withColumn("record_rank", F.row_number().over(window_spec))
)

display(
    ranked_df
    .filter(F.col("record_rank") > 1)
    .limit(20)
)

source,company,ingestion_timestamp,ingestion_date,job_id,job_title,department,office,team,job_level,employment_type,location,location_normalized,all_locations,workplace_type,is_remote,description,posted_at,updated_at,salary_min,salary_max,salary_currency,salary_interval,job_url,apply_url,quality_status,quality_reason,dq_processed_at,record_rank


In [0]:
# ============================================================
# KEEP BEST RECORD
# ============================================================

deduplicated_jobs = (
    ranked_df
    .filter(F.col("record_rank") == 1)
    .drop("record_rank")
)

print("Records after source-level deduplication:")
print(deduplicated_jobs.count())

Records after source-level deduplication:
2221


In [0]:
# ============================================================
# POST-DEDUPLICATION VALIDATION
# ============================================================

remaining_duplicates = (
    deduplicated_jobs
    .groupBy("source", "job_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Remaining duplicate source + job_id groups:",
    remaining_duplicates.count()
)

display(remaining_duplicates)

Remaining duplicate source + job_id groups: 0


source,job_id,count


In [0]:
from pyspark.sql import functions as F

# Read DQ-passed records
df = spark.table("job_market.silver.jobs_validated")

print("Validated records:", df.count())

Validated records: 2221


In [0]:
# ============================================================
# 1. DEDUPLICATE WITHIN EACH SOURCE
# ============================================================

from pyspark.sql.window import Window

window_spec = (
    Window
    .partitionBy("source", "job_id")
    .orderBy(
        F.col("posted_at").desc_nulls_last(),
        F.col("updated_at").desc_nulls_last(),
        F.col("ingestion_timestamp").desc_nulls_last()
    )
)

ranked_df = (
    df
    .withColumn("record_rank", F.row_number().over(window_spec))
)

deduplicated_jobs = (
    ranked_df
    .filter(F.col("record_rank") == 1)
    .drop("record_rank")
)

print("After source-level deduplication:",
      deduplicated_jobs.count())


# ============================================================
# 2. CROSS-SOURCE DUPLICATE ANALYSIS
# ============================================================

dedup_check = (
    deduplicated_jobs
    .withColumn(
        "business_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.lower(F.trim(F.col("company"))),
                F.lower(F.trim(F.col("job_title"))),
                F.lower(
                    F.trim(
                        F.coalesce(
                            F.col("location_normalized"),
                            F.lit("")
                        )
                    )
                )
            ),
            256
        )
    )
)

print("Business-key analysis created.")


# ============================================================
# 3. IDENTIFY POTENTIAL CROSS-SOURCE DUPLICATES
# ============================================================

cross_source_duplicates = (
    dedup_check
    .groupBy("business_key")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("source").alias("source_count"),
        F.collect_set("source").alias("sources")
    )
    .filter(
        (F.col("record_count") > 1) &
        (F.col("source_count") > 1)
    )
    .orderBy(F.desc("record_count"))
)

print(
    "Potential cross-source duplicate groups:",
    cross_source_duplicates.count()
)

display(cross_source_duplicates)


# ============================================================
# 4. CREATE FINAL CLEAN SILVER DATASET
# ============================================================

clean_jobs = (
    dedup_check
    .drop("business_key")
)

print(
    "Final clean Silver records:",
    clean_jobs.count()
)


# ============================================================
# 5. SAVE CLEAN SILVER TABLE
# ============================================================

clean_jobs.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("job_market.silver.jobs_clean")

print("✓ Created job_market.silver.jobs_clean")

After source-level deduplication: 2221
Business-key analysis created.
Potential cross-source duplicate groups: 0


business_key,record_count,source_count,sources


Final clean Silver records: 2221
✓ Created job_market.silver.jobs_clean


In [0]:
# ============================================================
# FINAL VALIDATION
# ============================================================

final_df = spark.table("job_market.silver.jobs_clean")

validated_count = spark.table(
    "job_market.silver.jobs_validated"
).count()

clean_count = final_df.count()

duplicates_removed = validated_count - clean_count

print("======================================")
print("DEDUPLICATION SUMMARY")
print("======================================")
print("Validated records  :", validated_count)
print("Clean records      :", clean_count)
print("Duplicates removed :", duplicates_removed)
print("======================================")


# Final uniqueness check
final_duplicates = (
    final_df
    .groupBy("source", "job_id")
    .count()
    .filter(F.col("count") > 1)
)

remaining_duplicates = final_duplicates.count()

print(
    "Remaining duplicate source + job_id groups:",
    remaining_duplicates
)

assert remaining_duplicates == 0, \
    "Duplicate source + job_id records still exist!"

print("✓ Deduplication validation passed.")

DEDUPLICATION SUMMARY
Validated records  : 2221
Clean records      : 2221
Duplicates removed : 0
Remaining duplicate source + job_id groups: 0
✓ Deduplication validation passed.
